# DP2 — DiaObject Catalog Query over a Deep Drilling Field

**Author:** dagoret  
**Date:** 2026-03-13  

## Purpose

This notebook queries the **DiaObject** catalog of Rubin DP2 DRP for a user-selected Deep Drilling
Field (DDF). It demonstrates the two complementary access methods:

- **TAP (Table Access Protocol)** — recommended for catalog-level cone searches; uses ADQL spatial
  constraints optimised for Qserv.
- **Butler Gen3** — recommended when image access is also needed, or when working with per-tract
  parquet tables.

The notebook then filters on `nDiaSources >= MIN_NSOURCES` (default 50), displays summary statistics,
plots spatial distributions and key variability diagnostics, and prepares the filtered table for
downstream cross-matching with external alert brokers (e.g. Fink).

## Coordinate-based cross-matching with Fink alerts

Because the Rubin Alert Production pipeline and the DRP (Data Release Production) are **independent
pipelines**, the `diaObjectId` values they assign to the same physical source are generally
**different and cannot be used to cross-identify objects across pipelines**.  
The correct approach is a **sky-coordinate cross-match** using (RA, Dec) + a matching radius,
as shown in Section 7.

## Data products used

| Product | Table / Dataset type | Description |
|---|---|---|
| DiaObject | `dp2.DiaObject` / `dia_object` | Summary statistics per transient / variable object |
| DiaSource | `dp2.DiaSource` / `dia_source` | Individual ≥5σ detections in difference images |
| ForcedSourceOnDiaObject | `dp2.ForcedSourceOnDiaObject` / `dia_object_forced_source` | Forced PSF photometry at DiaObject coordinates |

## References

- DP1 tutorial 201.4 — DiaObject table: https://dp1.lsst.io/tutorials/notebook/201/notebook-201-4.html  
- DP1 tutorial 306.2 — Candidate transient identification: https://dp1.lsst.io/tutorials/notebook/306/notebook-306-2.html  
- DP0.2 tutorial DP02_07a — DiaObject samples: https://dp0-2.lsst.io/_static/nb_html/DP02_07a_DiaObject_Samples.html  
- DDF coordinates reference: `2026-03-11_DP2_SurveyPropertyMaps_AllBands_AllDDF.ipynb`  
- SkyMap / tract reference: `2026-03-13_DP2_DDF_Tracts_SkyMap_Mollweide.ipynb`

---
## 0. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

# Astropy
from astropy.coordinates import SkyCoord
from astropy.table import Table
import astropy.units as u

# LSST Science Pipelines — Butler
import lsst.daf.butler as dafButler
import lsst.geom as geom

# Rubin Science Platform — TAP service
# (only available on the RSP; comment out if running offline)
from lsst.rsp import get_tap_service

print(f"Matplotlib : {matplotlib.__version__}")
print(f"Seaborn    : {sns.__version__}")
print(f"NumPy      : {np.__version__}")
print("Imports OK")

In [ ]:
#import ipywidgets as widgets
#%matplotlib widget

---
## 1. User Configuration

**Edit this cell** to choose the DDF, Butler repository, and query parameters.

In [ ]:
# =========================================================================
# USER PARAMETERS — edit here
# =========================================================================

# --- Target DDF ---
# Choose one key from DDF_COORDS below.
# Available: "COSMOS", "ECDFS", "ELAIS-S1", "XMM-LSS",
#            "EDFS-a", "EDFS-b", "EDFS", "M49"
SELECTED_DDF = "ECDFS"

# --- Search cone radius around the DDF center (degrees) ---
# This radius should encompass all tracts that cover the DDF.
# Use ~1.8° to match the tract search radius in the SkyMap notebook.
CONE_RADIUS_DEG = 1.8

# --- Minimum number of DiaSources per DiaObject ---
# Objects with fewer detections are likely artifacts or single-epoch events.
MIN_NSOURCES = 50

# --- Butler configuration ---
REPO        = "dp2_prep"           # Butler repository alias on the RSP
COLLECTION  = "LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545"
SKYMAP_NAME = "lsst_cells_v2"

# --- TAP catalog schema prefix ---
# For DP2 use "dp2"; for DP0.2 simulations use "dp02_dc2_catalogs"
TAP_SCHEMA = "dp2"

# =========================================================================
# DDF coordinate dictionary (RA, Dec in degrees)
# Source: 2026-03-11_DP2_SurveyPropertyMaps_AllBands_AllDDF.ipynb
# =========================================================================
DDF_COORDS = {
    "COSMOS"   : (150.119, +2.206),
    "ECDFS"    : (53.125,  -28.100),   # Extended Chandra Deep Field South
    "ELAIS-S1" : (9.450,   -44.000),
    "XMM-LSS"  : (35.708,  -4.750),
    "EDFS-a"   : (58.900,  -49.315),   # Euclid Deep Field South (a)
    "EDFS-b"   : (63.600,  -47.600),   # Euclid Deep Field South (b)
    "EDFS"     : (61.240,  -48.423),
    "M49"      : (187.400, +8.000),
}

# Resolve the selected DDF coordinates
if SELECTED_DDF not in DDF_COORDS:
    raise ValueError(f"Unknown DDF: '{SELECTED_DDF}'. "
                     f"Available: {list(DDF_COORDS.keys())}")

RA_CENTER, DEC_CENTER = DDF_COORDS[SELECTED_DDF]

print(f"Selected DDF   : {SELECTED_DDF}")
print(f"Center RA      : {RA_CENTER:.4f}°")
print(f"Center Dec     : {DEC_CENTER:+.4f}°")
print(f"Search radius  : {CONE_RADIUS_DEG}°")
print(f"Min nDiaSources: {MIN_NSOURCES}")
print(f"Butler repo    : {REPO}")
print(f"Collection     : {COLLECTION}")
print(f"TAP schema     : {TAP_SCHEMA}")

---
## 2. Method A — TAP Query (recommended for catalog-level access)

The TAP service queries the Qserv distributed catalog engine via ADQL.  
Always use `CONTAINS(POINT(...), CIRCLE(...))` rather than RA/Dec range constraints:
Qserv shards data by coordinate and spatial queries are typically much faster.

### 2.1 Start the TAP service

In [ ]:
# Start the TAP service — requires the Rubin Science Platform environment
service = get_tap_service("tap")
assert service is not None, "TAP service could not be started."
print("TAP service started:", service.baseurl)

### 2.2 Explore the DiaObject table schema

In [ ]:
# Retrieve the list of columns available in the DiaObject table
# Useful for adapting the SELECT clause to the actual data release schema.
schema_query = (
    f"SELECT column_name, description, unit, datatype "
    f"FROM TAP_SCHEMA.columns "
    f"WHERE table_name = '{TAP_SCHEMA}.DiaObject' "
    f"ORDER BY column_name ASC"
)
schema_result = service.search(schema_query).to_table()
print(f"DiaObject table: {len(schema_result)} columns")
schema_df = schema_result.to_pandas()
display(schema_df.head(40))

In [ ]:
# List all per-filter flux summary columns for quick reference
# DP2/DP1 follow the pattern: {band}_psfFlux{Stat}
flux_cols = schema_df[schema_df['column_name'].str.contains('psfFlux', na=False)]
print("Per-filter PSF-flux summary columns:")
print(flux_cols[['column_name', 'unit', 'description']].to_string(index=False))

### 2.3 Cone search — all DiaObjects in the DDF field

In [ ]:
# -------------------------------------------------------------------------
# ADQL cone search on the DiaObject table.
#
# Columns retrieved:
#   diaObjectId     : unique identifier in the DRP pipeline
#   ra, dec         : sky position (degrees, ICRS J2000)
#   nDiaSources     : total number of ≥5σ difference-image detections (all bands)
#   {b}_psfFluxMean : mean PSF flux in each band b (nJy)
#   {b}_psfFluxSigma: standard deviation of PSF fluxes (nJy)
#   {b}_psfFluxMax  : maximum PSF flux in each band (nJy)
#   {b}_psfFluxMin  : minimum PSF flux in each band (nJy)
#   {b}_psfFluxNdata: number of data points used for flux statistics in band b
#   {b}_psfFluxLinearSlope: linear slope of the light curve in band b (nJy/day)
#   {b}_scienceFluxMean: mean PSF flux in the science (direct) images (nJy)
#
# Note: column names may differ between DP0.2 and DP1/DP2.
# Adjust the SELECT clause if needed after inspecting the schema above.
# -------------------------------------------------------------------------

# Build the list of per-filter columns to retrieve
BANDS = ['u', 'g', 'r', 'i', 'z', 'y']
per_filter_cols = ", ".join(
    f"{b}_psfFluxMean, {b}_psfFluxSigma, {b}_psfFluxMax, "
    f"{b}_psfFluxMin, {b}_psfFluxNdata, {b}_psfFluxLinearSlope, "
    f"{b}_scienceFluxMean"
    for b in BANDS
)

query_all = (
    f"SELECT diaObjectId, ra, dec, nDiaSources, "
    f"{per_filter_cols} "
    f"FROM {TAP_SCHEMA}.DiaObject "
    f"WHERE CONTAINS("
    f"  POINT('ICRS', ra, dec), "
    f"  CIRCLE('ICRS', {RA_CENTER}, {DEC_CENTER}, {CONE_RADIUS_DEG})"
    f") = 1 "
    f"ORDER BY nDiaSources DESC"
)

print(f"Submitting TAP query for {SELECTED_DDF} (cone radius {CONE_RADIUS_DEG}°)...")
print(f"ADQL:\n{query_all}\n")

# Use an async job for potentially large result sets
job = service.submit_job(query_all)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
print(f"Job phase: {job.phase}")
if job.phase == 'ERROR':
    job.raise_if_error()
assert job.phase == 'COMPLETED'

dia_obj_all = job.fetch_result().to_table()
print(f"Total DiaObjects retrieved : {len(dia_obj_all):,}")

### 2.4 Filter on nDiaSources ≥ MIN_NSOURCES

In [ ]:
# Convert to pandas for easier filtering and display
df_all = dia_obj_all.to_pandas()

# Apply the minimum source count filter
df_rich = df_all[df_all['nDiaSources'] >= MIN_NSOURCES].copy()
df_rich = df_rich.sort_values('nDiaSources', ascending=False).reset_index(drop=True)

print(f"DiaObjects with nDiaSources >= {MIN_NSOURCES}: {len(df_rich):,}")
print(f"  Fraction of total: {len(df_rich)/len(df_all)*100:.2f}%")
print(f"  Max nDiaSources  : {df_rich['nDiaSources'].max() if len(df_rich) else 'N/A'}")
print(f"  Median nDiaSources: {df_rich['nDiaSources'].median() if len(df_rich) else 'N/A'}")

In [ ]:
# Display the top 20 most-detected DiaObjects
display_cols = ['diaObjectId', 'ra', 'dec', 'nDiaSources',
                'r_psfFluxMean', 'r_psfFluxSigma', 'r_psfFluxMax',
                'r_psfFluxMin', 'r_psfFluxNdata']
# Keep only columns that actually exist in the result
display_cols = [c for c in display_cols if c in df_rich.columns]

print(f"Top 20 DiaObjects with the most DiaSources ({SELECTED_DDF}):")
display(df_rich[display_cols].head(20))

---
## 3. Statistics and Diagnostic Plots

In [ ]:
# =========================================================================
# Figure 1 — Distribution of nDiaSources for all objects vs filtered sample
# =========================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: full distribution (log y-scale)
ax = axes[0]
ax.hist(df_all['nDiaSources'], bins=100, log=True,
        color='steelblue', alpha=0.8, edgecolor='white', linewidth=0.3)
ax.axvline(MIN_NSOURCES, color='red', linestyle='--', linewidth=1.5,
           label=f'Cut: nDiaSources ≥ {MIN_NSOURCES}')
ax.set_xlabel('nDiaSources (all bands)', fontsize=12)
ax.set_ylabel('Number of DiaObjects', fontsize=12)
ax.set_title(f'{SELECTED_DDF} — All DiaObjects\n(N = {len(df_all):,})',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Right: filtered distribution
ax = axes[1]
if len(df_rich) > 0:
    ax.hist(df_rich['nDiaSources'], bins=50, log=False,
            color='darkorange', alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.set_title(f'{SELECTED_DDF} — DiaObjects with nDiaSources ≥ {MIN_NSOURCES}\n'
                 f'(N = {len(df_rich):,})',
                 fontsize=12, fontweight='bold')
else:
    ax.text(0.5, 0.5, 'No objects pass the cut',
            transform=ax.transAxes, ha='center', va='center', fontsize=14)
    ax.set_title(f'{SELECTED_DDF} — Filtered sample (empty)', fontsize=12)
ax.set_xlabel('nDiaSources (all bands)', fontsize=12)
ax.set_ylabel('Number of DiaObjects', fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'DiaObj_nSources_hist_{SELECTED_DDF}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# =========================================================================
# Figure 2 — Sky map of DiaObjects (RA/Dec scatter)
# All objects: grey dots; filtered sample: colored by nDiaSources
# =========================================================================

fig, ax = plt.subplots(figsize=(9, 8))
ax.set_facecolor('#f5f5f5')
ax.invert_xaxis()    # RA increases towards the left by convention

# All DiaObjects (background, small grey points)
ax.scatter(df_all['ra'], df_all['dec'],
           s=1, color='lightgrey', alpha=0.4, label='All DiaObjects', zorder=1)

# Filtered DiaObjects — color-coded by nDiaSources
if len(df_rich) > 0:
    sc = ax.scatter(df_rich['ra'], df_rich['dec'],
                    c=df_rich['nDiaSources'],
                    cmap='plasma', s=12, alpha=0.85,
                    norm=mcolors.LogNorm(
                        vmin=df_rich['nDiaSources'].min(),
                        vmax=df_rich['nDiaSources'].max()),
                    zorder=3,
                    label=f'nDiaSources ≥ {MIN_NSOURCES}')
    cbar = plt.colorbar(sc, ax=ax, pad=0.02)
    cbar.set_label('nDiaSources (log scale)', fontsize=10)

# DDF center marker
ax.plot(RA_CENTER, DEC_CENTER, marker='*', markersize=18,
        color='gold', markeredgecolor='black', markeredgewidth=1.0,
        zorder=10, linestyle='none', label=f'{SELECTED_DDF} center')

# Search cone boundary (approximate circle for display)
cos_dec = np.cos(np.deg2rad(DEC_CENTER))
theta   = np.linspace(0, 2 * np.pi, 360)
ax.plot(RA_CENTER + CONE_RADIUS_DEG / cos_dec * np.cos(theta),
        DEC_CENTER + CONE_RADIUS_DEG * np.sin(theta),
        color='red', linewidth=1.2, linestyle='--',
        label=f'Search cone ({CONE_RADIUS_DEG}°)', zorder=4)

ax.set_xlabel('Right Ascension (deg)', fontsize=12)
ax.set_ylabel('Declination (deg)', fontsize=12)
ax.set_title(f'DP2 DiaObjects — {SELECTED_DDF}\n'
             f'Total: {len(df_all):,}   Filtered (nDiaSrc ≥ {MIN_NSOURCES}): {len(df_rich):,}',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig(f'DiaObj_skymap_{SELECTED_DDF}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# =========================================================================
# Figure 3 — Variability diagnostics for the filtered sample (r-band)
# =========================================================================

if len(df_rich) == 0:
    print("No objects in the filtered sample — skipping variability plots.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # --- Panel A: Mean vs sigma (r-band) ---
    ax = axes[0, 0]
    if 'r_psfFluxMean' in df_rich.columns and 'r_psfFluxSigma' in df_rich.columns:
        mask = (df_rich['r_psfFluxMean'] > 0) & (df_rich['r_psfFluxSigma'] > 0)
        ax.scatter(df_rich.loc[mask, 'r_psfFluxMean'],
                   df_rich.loc[mask, 'r_psfFluxSigma'],
                   s=4, alpha=0.6, color='steelblue')
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('r-band mean PSF flux (nJy)', fontsize=11)
        ax.set_ylabel('r-band flux std dev (nJy)', fontsize=11)
        ax.set_title('r-band mean vs σ flux', fontsize=11)
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'r_psfFluxMean / r_psfFluxSigma not available',
                transform=ax.transAxes, ha='center', va='center')

    # --- Panel B: Flux amplitude (r_psfFluxMax - r_psfFluxMin) histogram ---
    ax = axes[0, 1]
    if 'r_psfFluxMax' in df_rich.columns and 'r_psfFluxMin' in df_rich.columns:
        amp = df_rich['r_psfFluxMax'] - df_rich['r_psfFluxMin']
        ax.hist(amp[np.isfinite(amp)], bins=60, color='darkorange',
                alpha=0.8, edgecolor='white', linewidth=0.3)
        ax.set_xlabel('r-band flux amplitude: max − min (nJy)', fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title('r-band flux amplitude distribution', fontsize=11)
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'r_psfFluxMax / r_psfFluxMin not available',
                transform=ax.transAxes, ha='center', va='center')

    # --- Panel C: Linear slope of r-band light curve ---
    ax = axes[1, 0]
    if 'r_psfFluxLinearSlope' in df_rich.columns:
        slope = df_rich['r_psfFluxLinearSlope'].dropna()
        # Clip extreme outliers for display
        p1, p99 = np.nanpercentile(slope, [1, 99])
        ax.hist(slope.clip(p1, p99), bins=60, color='mediumseagreen',
                alpha=0.8, edgecolor='white', linewidth=0.3)
        ax.axvline(0, color='red', linestyle='--', linewidth=1.2)
        ax.set_xlabel('r-band PSF flux linear slope (nJy/day)', fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title('r-band light curve slope\n(negative = fading, positive = brightening)',
                     fontsize=11)
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'r_psfFluxLinearSlope not available',
                transform=ax.transAxes, ha='center', va='center')

    # --- Panel D: nDiaSources per band (r_psfFluxNdata) ---
    ax = axes[1, 1]
    ndata_cols = [f'{b}_psfFluxNdata' for b in BANDS if f'{b}_psfFluxNdata' in df_rich.columns]
    if ndata_cols:
        ndata_means = [df_rich[c].mean() for c in ndata_cols]
        band_labels = [c.split('_')[0] for c in ndata_cols]
        palette = sns.color_palette("tab10", len(band_labels))
        ax.bar(band_labels, ndata_means, color=palette, edgecolor='white', linewidth=0.5)
        ax.set_xlabel('Band', fontsize=11)
        ax.set_ylabel('Mean nDiaSources per band', fontsize=11)
        ax.set_title('Mean number of r-band detections\nper DiaObject (filtered sample)',
                     fontsize=11)
        ax.grid(True, alpha=0.3, axis='y')
    else:
        ax.text(0.5, 0.5, 'Per-band nData columns not available',
                transform=ax.transAxes, ha='center', va='center')

    fig.suptitle(
        f'DP2 DiaObject variability diagnostics — {SELECTED_DDF}\n'
        f'(nDiaSources ≥ {MIN_NSOURCES},  N = {len(df_rich):,})',
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig(f'DiaObj_variability_{SELECTED_DDF}.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 4. Method B — Butler Access (alternative / supplement)

The Butler stores the DiaObject catalog as per-tract parquet files (`dia_object` dataset type).
This is more cumbersome for a simple cone search but is mandatory when combining catalog and
image access within the same notebook.

Butler dataset type: `('dia_object', {skymap, tract}, ArrowAstropy)`  
DiaSource:          `('dia_source', {skymap, tract}, ArrowAstropy)`  
ForcedSource:       `('dia_object_forced_source', {skymap, tract, patch}, ArrowAstropy)`

In [ ]:
# -------------------------------------------------------------------------
# Instantiate the Butler and load the SkyMap
# -------------------------------------------------------------------------
butler = dafButler.Butler(REPO, collections=COLLECTION)
skymap = butler.get("skyMap", skymap=SKYMAP_NAME)

print(f"Butler instantiated  : {REPO}")
print(f"Collection           : {COLLECTION}")
print(f"SkyMap               : {SKYMAP_NAME}")
print(f"Number of tracts     : {len(skymap)}")

In [ ]:
# -------------------------------------------------------------------------
# Find the tracts that cover the selected DDF center
# (same approach as in 2026-03-13_DP2_DDF_Tracts_SkyMap_Mollweide.ipynb)
# -------------------------------------------------------------------------

def find_tracts_for_coord(skymap, ra_deg, dec_deg, radius_deg=1.8):
    """
    Return deduplicated list of TractInfo whose footprint overlaps
    the disk (ra_deg, dec_deg, radius_deg) using a grid-sampling approach.
    """
    cos_dec = max(np.cos(np.deg2rad(dec_deg)), 0.01)
    step    = 0.35   # grid step in degrees
    found_ids = set()
    for ddec in np.arange(-radius_deg, radius_deg + step, step):
        for dra in np.arange(-radius_deg, radius_deg + step, step):
            if np.sqrt(dra**2 + ddec**2) > radius_deg:
                continue
            ra_s  = ra_deg  + dra / cos_dec
            dec_s = dec_deg + ddec
            if not (-89.9 <= dec_s <= 89.9):
                continue
            sp = geom.SpherePoint(ra_s * geom.degrees, dec_s * geom.degrees)
            try:
                ti = skymap.findTract(sp)
                found_ids.add(ti.tract_id)
            except Exception:
                pass
    return [skymap[tid] for tid in sorted(found_ids)]


ddf_tracts = find_tracts_for_coord(skymap, RA_CENTER, DEC_CENTER,
                                   radius_deg=CONE_RADIUS_DEG)
tract_ids  = [t.tract_id for t in ddf_tracts]
print(f"Tracts covering {SELECTED_DDF}: {tract_ids}")

In [ ]:
# -------------------------------------------------------------------------
# Load the dia_object table for each tract via the Butler
# and concatenate into a single Astropy table.
#
# The dataset type 'dia_object' requires dataId keys: {skymap, tract}
# -------------------------------------------------------------------------

tables = []
for tract_id in tract_ids:
    dataId = {'skymap': SKYMAP_NAME, 'tract': tract_id}
    try:
        t = butler.get('dia_object', dataId=dataId)
        # t is an ArrowAstropy table; convert to astropy Table
        if hasattr(t, 'to_astropy'):
            t = t.to_astropy()
        tables.append(t)
        print(f"  Tract {tract_id}: {len(t):,} DiaObjects loaded.")
    except Exception as exc:
        print(f"  Tract {tract_id}: could not load dia_object — {exc}")

if tables:
    from astropy.table import vstack
    dia_obj_butler = vstack(tables)
    print(f"\nTotal DiaObjects from Butler: {len(dia_obj_butler):,}")
    print(f"Columns: {dia_obj_butler.colnames[:20]} ...")
else:
    print("No DiaObject data loaded from Butler.")
    dia_obj_butler = None

In [ ]:
# -------------------------------------------------------------------------
# Apply cone filter and nDiaSources cut on the Butler result
# (the Butler loads the full tract, so we must apply the spatial cut manually)
# -------------------------------------------------------------------------

if dia_obj_butler is not None:
    df_butler = dia_obj_butler.to_pandas()

    # Spatial filter: keep only objects within CONE_RADIUS_DEG of the DDF center
    center_sky = SkyCoord(ra=RA_CENTER * u.deg, dec=DEC_CENTER * u.deg)
    obj_sky    = SkyCoord(ra=df_butler['ra'].values  * u.deg,
                          dec=df_butler['dec'].values * u.deg)
    sep_deg    = center_sky.separation(obj_sky).deg

    in_cone     = sep_deg <= CONE_RADIUS_DEG
    df_butler_cone = df_butler[in_cone].copy()

    # nDiaSources filter
    nsrc_col = 'nDiaSources' if 'nDiaSources' in df_butler_cone.columns else None
    if nsrc_col:
        df_butler_rich = df_butler_cone[
            df_butler_cone[nsrc_col] >= MIN_NSOURCES
        ].copy()
        df_butler_rich = df_butler_rich.sort_values(nsrc_col, ascending=False)
        print(f"Butler: objects in cone                 : {len(df_butler_cone):,}")
        print(f"Butler: objects with nDiaSources >= {MIN_NSOURCES:3d} : {len(df_butler_rich):,}")
        display(df_butler_rich.head(10))
    else:
        print("Column 'nDiaSources' not found — check schema.")
        df_butler_rich = df_butler_cone.copy()
else:
    print("Butler result not available.")
    df_butler_rich = None

---
## 5. Retrieve DiaSources for the Filtered DiaObjects (TAP)

For each DiaObject that passes the `nDiaSources >= MIN_NSOURCES` cut, retrieve all individual
DiaSource detections from the `DiaSource` table.  
This gives access to per-epoch fluxes, timestamps (`midPointMjdTai`), and filter information.

In [ ]:
# -------------------------------------------------------------------------
# Retrieve DiaSources for the filtered DiaObjects via TAP
#
# Strategy: use a second cone query on DiaSource (same cone as DiaObject),
# then JOIN in Python by diaObjectId.
#
# Note: Qserv does not support sub-queries, so we cannot do
#   WHERE diaObjectId IN (...) for large ID lists.
#   A spatial cone on DiaSource is the recommended approach.
# -------------------------------------------------------------------------

query_diasrc = (
    f"SELECT diaSourceId, diaObjectId, ra, dec, "
    f"midPointMjdTai, band, "
    f"psfFlux, psfFluxErr, "
    f"apFlux, apFluxErr, "
    f"snr, "
    f"ccdVisitId "
    f"FROM {TAP_SCHEMA}.DiaSource "
    f"WHERE CONTAINS("
    f"  POINT('ICRS', ra, dec), "
    f"  CIRCLE('ICRS', {RA_CENTER}, {DEC_CENTER}, {CONE_RADIUS_DEG})"
    f") = 1"
)

print("Submitting TAP query for DiaSources...")
print(f"ADQL:\n{query_diasrc}\n")

job_src = service.submit_job(query_diasrc)
job_src.run()
job_src.wait(phases=['COMPLETED', 'ERROR'])
print(f"Job phase: {job_src.phase}")
if job_src.phase == 'ERROR':
    job_src.raise_if_error()
assert job_src.phase == 'COMPLETED'

dia_src_all = job_src.fetch_result().to_table()
df_src_all  = dia_src_all.to_pandas()
print(f"Total DiaSources in cone: {len(df_src_all):,}")

In [ ]:
# -------------------------------------------------------------------------
# Keep only DiaSources associated with the filtered DiaObjects
# (inner join by diaObjectId)
# -------------------------------------------------------------------------

rich_ids    = set(df_rich['diaObjectId'].values)
df_src_rich = df_src_all[df_src_all['diaObjectId'].isin(rich_ids)].copy()

print(f"DiaSources linked to objects with nDiaSources >= {MIN_NSOURCES}: "
      f"{len(df_src_rich):,}")
display(df_src_rich.head(10))

In [ ]:
# =========================================================================
# Figure 4 — MJD coverage of DiaSources (all bands)
# =========================================================================

if len(df_src_rich) > 0 and 'midPointMjdTai' in df_src_rich.columns:
    fig, ax = plt.subplots(figsize=(13, 4))
    band_col   = 'band' if 'band' in df_src_rich.columns else 'filterName'
    band_colors = {'u': 'purple', 'g': 'green', 'r': 'red',
                   'i': 'darkorange', 'z': 'brown', 'y': 'black'}

    for band, grp in df_src_rich.groupby(band_col):
        ax.scatter(grp['midPointMjdTai'], grp['psfFlux'],
                   s=1, alpha=0.25,
                   color=band_colors.get(band, 'grey'),
                   label=band)

    ax.set_xlabel('MJD (TAI)', fontsize=12)
    ax.set_ylabel('PSF flux (nJy)', fontsize=12)
    ax.set_title(f'All DiaSources of filtered DiaObjects — {SELECTED_DDF}\n'
                 f'(nDiaSources ≥ {MIN_NSOURCES},  N_src = {len(df_src_rich):,})',
                 fontsize=12, fontweight='bold')
    ax.legend(title='Band', fontsize=9, markerscale=6)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'DiaObj_lightcurves_{SELECTED_DDF}.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Insufficient data for MJD plot.")

---
## 6. Forced Photometry — ForcedSourceOnDiaObject (TAP)

`ForcedSourceOnDiaObject` provides forced PSF photometry at the position of **every DiaObject**,
in **every visit image** (not just those with a ≥5σ detection).  
This is the preferred table for light curve analysis because it recovers epochs where the source
fell below the detection threshold.

In [ ]:
# -------------------------------------------------------------------------
# Retrieve ForcedSourceOnDiaObject via TAP (spatial cone + JOIN to DiaObject)
#
# The table is joined to DiaObject to filter spatially.
# The JOIN uses diaObjectId as the key.
#
# Note: For very large fields this query can be slow; restrict to the
# filtered (rich) DiaObject set by using an additional nDiaSources filter
# directly in the JOIN.
# -------------------------------------------------------------------------

query_forced = (
    f"SELECT f.diaObjectId, f.ccdVisitId, "
    f"f.band, f.midpointMjdTai, "
    f"f.psfFlux, f.psfFluxErr, "
    f"f.psfDiffFlux, f.psfDiffFluxErr, "
    f"o.ra, o.dec, o.nDiaSources "
    f"FROM {TAP_SCHEMA}.ForcedSourceOnDiaObject AS f "
    f"JOIN {TAP_SCHEMA}.DiaObject AS o "
    f"  ON f.diaObjectId = o.diaObjectId "
    f"WHERE CONTAINS("
    f"  POINT('ICRS', o.ra, o.dec), "
    f"  CIRCLE('ICRS', {RA_CENTER}, {DEC_CENTER}, {CONE_RADIUS_DEG})"
    f") = 1 "
    f"AND o.nDiaSources >= {MIN_NSOURCES}"
)

print("Submitting TAP query for ForcedSourceOnDiaObject...")
print(f"ADQL:\n{query_forced}\n")

job_forced = service.submit_job(query_forced)
job_forced.run()
job_forced.wait(phases=['COMPLETED', 'ERROR'])
print(f"Job phase: {job_forced.phase}")
if job_forced.phase == 'ERROR':
    job_forced.raise_if_error()
assert job_forced.phase == 'COMPLETED'

df_forced = job_forced.fetch_result().to_table().to_pandas()
print(f"ForcedSource rows retrieved: {len(df_forced):,}")
display(df_forced.head(10))

In [ ]:
# =========================================================================
# Figure 5 — Example forced-photometry light curve for the most-detected object
# =========================================================================

if len(df_forced) > 0 and len(df_rich) > 0:
    # Select the DiaObject with the highest nDiaSources
    top_id = df_rich.iloc[0]['diaObjectId']
    top_n  = df_rich.iloc[0]['nDiaSources']
    df_lc  = df_forced[df_forced['diaObjectId'] == top_id].copy()
    df_lc  = df_lc.sort_values('midpointMjdTai')

    band_col_forced = 'band' if 'band' in df_lc.columns else 'filterName'
    band_colors = {'u': 'purple', 'g': 'green', 'r': 'red',
                   'i': 'darkorange', 'z': 'brown', 'y': 'black'}

    fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

    # --- Top panel: direct (science) image PSF flux ---
    ax = axes[0]
    for band, grp in df_lc.groupby(band_col_forced):
        ax.errorbar(grp['midpointMjdTai'], grp['psfFlux'],
                    yerr=grp['psfFluxErr'],
                    fmt='o', ms=3, alpha=0.7,
                    color=band_colors.get(band, 'grey'),
                    label=band, capsize=2)
    ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
    ax.set_ylabel('Science image PSF flux (nJy)', fontsize=11)
    ax.set_title(
        f'Forced-photometry light curve — diaObjectId = {top_id}\n'
        f'{SELECTED_DDF}  |  nDiaSources = {top_n}',
        fontsize=12, fontweight='bold'
    )
    ax.legend(title='Band', fontsize=9)
    ax.grid(True, alpha=0.3)

    # --- Bottom panel: difference image PSF flux ---
    ax = axes[1]
    diff_col = 'psfDiffFlux' if 'psfDiffFlux' in df_lc.columns else None
    if diff_col:
        for band, grp in df_lc.groupby(band_col_forced):
            ax.errorbar(grp['midpointMjdTai'], grp[diff_col],
                        yerr=grp['psfDiffFluxErr'],
                        fmt='o', ms=3, alpha=0.7,
                        color=band_colors.get(band, 'grey'),
                        label=band, capsize=2)
        ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
        ax.set_ylabel('Difference image PSF flux (nJy)', fontsize=11)
        ax.legend(title='Band', fontsize=9)
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'psfDiffFlux column not available',
                transform=ax.transAxes, ha='center')

    ax.set_xlabel('MJD (TAI)', fontsize=11)
    plt.tight_layout()
    plt.savefig(f'DiaObj_forcedLC_{top_id}_{SELECTED_DDF}.png',
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No forced photometry data available for light curve plot.")

---
## 7. Coordinate-based Cross-Match with an External Alert (e.g. Fink)

### Rationale

The **Rubin Alert Production** pipeline (which feeds brokers like Fink) and the **DRP** (Data
Release Production) pipeline are **independent**.  They process the same raw images through
different software versions and configurations, and they assign **different `diaObjectId` values**
to the same physical source.  There is no reliable mapping between the two IDs.

The only robust way to identify the DRP counterpart of a Fink alert is to **cross-match by
sky coordinates** using a matching radius commensurate with the astrometric uncertainty
(typically 0.5 – 2 arcseconds for ground-based imaging).

### Workflow

1. Provide the (RA, Dec) of the Fink alert.
2. Query the DRP DiaObject catalog within a small radius around that position.
3. Rank candidates by angular separation.
4. Optionally refine using `nDiaSources`, flux amplitude, or temporal overlap.

In [ ]:
# =========================================================================
# User-supplied Fink alert coordinates
# Replace with the actual (RA, Dec) from the Fink alert broker.
# =========================================================================

# Example coordinates inside the ECDFS footprint (edit as needed)
FINK_RA  = 53.125    # degrees (ICRS J2000)
FINK_DEC = -28.100   # degrees (ICRS J2000)

# Cross-match radius (arcseconds)
XMATCH_RADIUS_ARCSEC = 1.0

print(f"Fink alert coordinates : RA={FINK_RA:.6f}°,  Dec={FINK_DEC:+.6f}°")
print(f"Cross-match radius     : {XMATCH_RADIUS_ARCSEC} arcsec")

In [ ]:
# -------------------------------------------------------------------------
# Step 1: TAP cone search — retrieve all DiaObjects within the cross-match radius
# Use a small cone (5× the matching radius) to keep the query fast.
# -------------------------------------------------------------------------

xmatch_cone_deg = max(XMATCH_RADIUS_ARCSEC / 3600.0 * 5, 10.0 / 3600.0)

query_xmatch = (
    f"SELECT diaObjectId, ra, dec, nDiaSources, "
    f"r_psfFluxMean, r_psfFluxSigma, r_psfFluxMax, r_psfFluxMin "
    f"FROM {TAP_SCHEMA}.DiaObject "
    f"WHERE CONTAINS("
    f"  POINT('ICRS', ra, dec), "
    f"  CIRCLE('ICRS', {FINK_RA}, {FINK_DEC}, {xmatch_cone_deg})"
    f") = 1"
)

print(f"Cross-match TAP cone: {xmatch_cone_deg*3600:.1f} arcsec")
xmatch_result = service.search(query_xmatch).to_table()
df_xmatch = xmatch_result.to_pandas()
print(f"DiaObjects found in TAP cone: {len(df_xmatch)}")

In [ ]:
# -------------------------------------------------------------------------
# Step 2: Compute angular separation and rank candidates
# -------------------------------------------------------------------------

if len(df_xmatch) == 0:
    print("No DiaObject found within the search cone — "
          "try increasing XMATCH_RADIUS_ARCSEC or verify coordinates.")
else:
    alert_sky = SkyCoord(ra=FINK_RA * u.deg, dec=FINK_DEC * u.deg)
    cand_sky  = SkyCoord(ra=df_xmatch['ra'].values  * u.deg,
                         dec=df_xmatch['dec'].values * u.deg)

    sep_arcsec = alert_sky.separation(cand_sky).arcsec
    df_xmatch  = df_xmatch.copy()
    df_xmatch['sep_arcsec'] = sep_arcsec

    # Apply the strict matching radius
    df_xmatch_cut = df_xmatch[
        df_xmatch['sep_arcsec'] <= XMATCH_RADIUS_ARCSEC
    ].sort_values('sep_arcsec').reset_index(drop=True)

    print(f"Candidates within {XMATCH_RADIUS_ARCSEC}\" : {len(df_xmatch_cut)}")

    if len(df_xmatch_cut) > 0:
        print("\nBest match(es):")
        display(df_xmatch_cut[['diaObjectId', 'ra', 'dec',
                                'sep_arcsec', 'nDiaSources',
                                'r_psfFluxMean', 'r_psfFluxSigma']].head(5))

        # The DRP diaObjectId of the best match
        best_drp_id  = df_xmatch_cut.iloc[0]['diaObjectId']
        best_sep     = df_xmatch_cut.iloc[0]['sep_arcsec']
        print(f"\nBest DRP DiaObject ID : {best_drp_id}")
        print(f"Angular separation    : {best_sep:.4f} arcsec")
    else:
        print("No match within the strict radius.")

In [ ]:
# -------------------------------------------------------------------------
# Step 3 (optional): Retrieve the forced-photometry light curve
# of the best DRP cross-match candidate
# -------------------------------------------------------------------------

if len(df_xmatch_cut) > 0:
    best_drp_id = df_xmatch_cut.iloc[0]['diaObjectId']
    best_ra     = df_xmatch_cut.iloc[0]['ra']
    best_dec    = df_xmatch_cut.iloc[0]['dec']

    # Query ForcedSourceOnDiaObject using a point search near the best candidate
    tiny_cone = 5.0 / 3600.0   # 5 arcsec cone to retrieve only the target object

    query_lc = (
        f"SELECT f.diaObjectId, f.ccdVisitId, f.band, f.midpointMjdTai, "
        f"f.psfFlux, f.psfFluxErr, f.psfDiffFlux, f.psfDiffFluxErr "
        f"FROM {TAP_SCHEMA}.ForcedSourceOnDiaObject AS f "
        f"JOIN {TAP_SCHEMA}.DiaObject AS o ON f.diaObjectId = o.diaObjectId "
        f"WHERE CONTAINS("
        f"  POINT('ICRS', o.ra, o.dec), "
        f"  CIRCLE('ICRS', {best_ra}, {best_dec}, {tiny_cone})"
        f") = 1"
    )

    print(f"Fetching forced-photometry light curve for diaObjectId = {best_drp_id}...")
    df_best_lc = service.search(query_lc).to_table().to_pandas()
    df_best_lc = df_best_lc.sort_values('midpointMjdTai')
    print(f"  Epochs retrieved: {len(df_best_lc)}")

    # ---- Plot ----
    if len(df_best_lc) > 0:
        band_colors = {'u': 'purple', 'g': 'green', 'r': 'red',
                       'i': 'darkorange', 'z': 'brown', 'y': 'black'}
        band_col_lc = 'band' if 'band' in df_best_lc.columns else 'filterName'

        fig, ax = plt.subplots(figsize=(12, 5))
        for band, grp in df_best_lc.groupby(band_col_lc):
            ax.errorbar(grp['midpointMjdTai'], grp['psfDiffFlux'],
                        yerr=grp['psfDiffFluxErr'],
                        fmt='o', ms=4, alpha=0.8,
                        color=band_colors.get(band, 'grey'),
                        label=band, capsize=2)

        ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
        ax.set_xlabel('MJD (TAI)', fontsize=12)
        ax.set_ylabel('Difference image PSF flux (nJy)', fontsize=12)
        ax.set_title(
            f'DRP forced-photometry light curve — cross-matched DiaObject\n'
            f'diaObjectId = {best_drp_id}   sep = {best_sep:.3f}"   "
            f'Fink alert: RA={FINK_RA:.5f}°  Dec={FINK_DEC:+.5f}°',
            fontsize=11, fontweight='bold'
        )
        ax.legend(title='Band', fontsize=9)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(f'DiaObj_xmatch_LC_{best_drp_id}.png', dpi=150, bbox_inches='tight')
        plt.show()
else:
    print("No cross-match candidate — light curve plot skipped.")

---
## 8. Save Results

Export the filtered DiaObject table and the matched DiaSources as CSV/FITS files.

In [ ]:
# -------------------------------------------------------------------------
# Save filtered DiaObject table (TAP result)
# -------------------------------------------------------------------------
outfile_obj = f"DiaObjects_{SELECTED_DDF}_nmin{MIN_NSOURCES}.csv"
df_rich.to_csv(outfile_obj, index=False)
print(f"Saved filtered DiaObject table : {outfile_obj}  ({len(df_rich):,} rows)")

# Save associated DiaSources
if len(df_src_rich) > 0:
    outfile_src = f"DiaSources_{SELECTED_DDF}_nmin{MIN_NSOURCES}.csv"
    df_src_rich.to_csv(outfile_src, index=False)
    print(f"Saved DiaSource table          : {outfile_src}  ({len(df_src_rich):,} rows)")

# Save forced photometry if available
if len(df_forced) > 0:
    outfile_forced = f"ForcedSrc_{SELECTED_DDF}_nmin{MIN_NSOURCES}.csv"
    df_forced.to_csv(outfile_forced, index=False)
    print(f"Saved ForcedSourceOnDiaObject  : {outfile_forced}  ({len(df_forced):,} rows)")

---
## Technical Summary

| Parameter | Value |
|---|---|
| Butler repository | `dp2_prep` |
| Collection | `LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545` |
| SkyMap | `lsst_cells_v2` |
| TAP schema | `dp2` |
| Selected DDF | configurable via `SELECTED_DDF` |
| Cone radius | configurable via `CONE_RADIUS_DEG` |
| Min nDiaSources | configurable via `MIN_NSOURCES` |
| Cross-match radius | configurable via `XMATCH_RADIUS_ARCSEC` |

### Key API notes

- The `diaObjectId` in the **DRP** and in **alert stream** brokers (Fink, ALeRCE, etc.) are
  assigned by independent pipelines and are **not cross-compatible**. Always cross-match by
  sky coordinates.
- Always use **ADQL spatial constraints** (`CONTAINS / CIRCLE`) with Qserv: range queries on
  RA/Dec (`BETWEEN`) bypass the spatial index and can be orders of magnitude slower.
- For light curves, prefer `ForcedSourceOnDiaObject` over `DiaSource`: it provides
  forced-photometry measurements at **every epoch**, not only those with an above-threshold
  detection.